In [11]:
from google import genai

In [12]:
client = genai.Client(api_key = "AQ.Ab8RN6LjY5W1cO4TZC3j-_GFKIcw2a46Rn8tawyv9yOQmZZiMQ")

In [14]:
def get_completion_from_messages(messages, model = "gemini-3.5-flash-lite", temperature = 0, max_tokens = 500):
    response = client.models.generate_content(model = model, 
                                              #contents = messages[-1]['content'],
                                              contents = "\n".join(
                                                                f"{msg['role']}: {msg['content']}"
                                                                for msg in messages),
                                              config = {'temperature': temperature,
                                                        'max_output_tokens': max_tokens}
                                             )
    return response.text.strip()

In [15]:
final_response_to_customer = f"""
The SmartX ProPhone has a 6.1-inch display, 128GB storage, \
12MP dual camera, and 5G. The FotoSnap DSLR Camera \
has a 24.2MP sensor, 1080p video, 3-inch LCD, and \
interchangeable lenses. We have a variety of TVs, including \
the CineView 4K TV with a 55-inch display, 4K resolution, \
HDR, and smart TV features. We also have the SoundMax \
Home Theater system with 5.1 channel, 1000W output, wireless \
subwoofer, and Bluetooth. Do you have any specific questions \
about these products or any other products we offer?
"""

In [16]:
# moderation using gemini 

moderation_prompt = f"""
you are a content moderator. analyze the following text for harmful content, hate speech, harassment, or unsafe topics.
Respond with a single letter:
Y - if it contain harmful content
N - if it is safe.

Text: '''{final_response_to_customer}'''
"""

moderation_response = get_completion_from_messages([{
    'role': 'user', 'content': moderation_prompt}])
print("Moderation Decision:", moderation_response)

Moderation Decision: N


In [17]:
system_message = f"""
You are an assistant that evaluates whether \
customer service agent responses sufficiently \
answer customer questions, and also validates that \
all the facts the assistant cites from the product \
information are correct.
The product information and user and customer \
service agent messages will be delimited by \
3 backticks, i.e. \`\`\`.
Respond with a Y or N character, with no punctuation:
Y - if the output sufficiently answers the question \
AND the response correctly uses product information
N - otherwise

Output a single letter only.
"""

customer_message = f"""
tell me about the smartx pro phone and \
the fotosnap camera, the dslr one. \
Also tell me about your tvs"""

product_information ="""{ "name": "SmartX ProPhone", "category": "Smartphones and Accessories",
"brand": "SmartX", "model_number": "SX-PP10", "warranty": "1 year",
"rating": 4.6, "features": [ "6.1-inch display", "128GB storage", "12MP dual camera", "5G" ],
"description": "A powerful smartphone with advanced camera features.", "price": 899.99 } { "name": "FotoSnap DSLR Camera",
"category": "Cameras and Camcorders", "brand": "FotoSnap", "model_number": "FS-DSLR200",
"warranty": "1 year", "rating": 4.7, "features": [ "24.2MP sensor", "1080p video", "3-inch LCD", "Interchangeable lenses" ],
"description": "Capture stunning photos and videos with this versatile DSLR camera.", "price": 599.99 } """

q_a_pair = f"""
Customer message: {customer_message}
Product Information: {product_information}
Agent response: {final_response_to_customer}
Does the response use the retrived information correctly?
Does the response sufficiently answer the question?

Output Y or N
"""

messages = [
    {'role':'system', 'content': system_message},
    {'role':'user', 'content': q_a_pair}
]

response = get_completion_from_messages(messages, max_tokens=5)
print(response)

N
